[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USER/REPO/blob/main/workshop_multiomics_integration.ipynb)

# Hands-On Multi-Omics: Integrating scRNA-seq and scATAC-seq
**Festival of Genomics Workshop — ~45 minutes**

In this workshop you will:
1. Look at scATAC-seq data and its quality metrics
2. See how the same cells look from two modalities (RNA + ATAC)
3. **Link individual chromatin peaks to individual genes** using paired multiome data — the central analysis
4. Use a deep-learning model (AlphaGenome) to predict which transcription factors bind a peak

**Dataset:** 10x Genomics PBMC 10k Multiome (paired scRNA-seq + scATAC-seq from the same cells).

**RNA annotation** has been done in advance — see [`rna_annotation.ipynb`](rna_annotation.ipynb) for the full walkthrough.

---

In [ ]:
%%capture
!pip install -q muon scanpy alphagenome leidenalg igv-notebook

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import muon as mu
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(6, 5))
sc.settings.verbosity = 1

## Introduction: what is a multiome experiment? (~5 min)

*[Insert a multiome schematic here for the live presentation.]*

**Single-cell multiome** measures two modalities **from the same nucleus**:

- **scRNA-seq** — which genes the cell is making (mRNA counts per gene)
- **scATAC-seq** — which regions of the genome are *accessible* (open chromatin), i.e. where transcription factors can bind and where regulation is happening

Each cell yields one row in an RNA matrix *and* one row in an ATAC matrix, linked by a shared barcode. The two views are perfectly aligned: row 1 in RNA and row 1 in ATAC are the same physical cell.

**Why this matters.** Single-modality scRNA tells us what cells *are*. Multiome adds **which regulatory regions are active in each cell** — and because the rows are paired, we can ask the central question of gene regulation: *when this peak opens, does this gene's expression change?*

The 10x dataset we'll use here is from cryopreserved human PBMCs (peripheral blood mononuclear cells).

In [ ]:
# Load workshop data
# Place pbmc_10k_multiome_workshop.h5mu in the working directory.
# Optionally set DATA_URL env var to auto-download from a remote bucket.
import os, urllib.request

DATA_FILE = "pbmc_10k_multiome_workshop.h5mu"
DATA_URL = os.environ.get("DATA_URL")  # e.g. "https://.../pbmc_10k_multiome_workshop.h5mu"

if not os.path.exists(DATA_FILE):
    if DATA_URL:
        print(f"Downloading from {DATA_URL} ...")
        urllib.request.urlretrieve(DATA_URL, DATA_FILE)
        print("Done!")
    else:
        raise FileNotFoundError(
            f"{DATA_FILE} not found. Run preprocessing/preprocess_pbmc_multiome.py "
            f"to generate it, or set DATA_URL to a hosted copy."
        )

mdata = mu.read(DATA_FILE)
mdata

## What's in the `.h5mu`

`MuData` holds two AnnData objects side-by-side (one per modality), plus shared metadata:

- `mdata.mod['rna']` — cells × genes, log-normalized, with `cell_type` already annotated (see [`rna_annotation.ipynb`](rna_annotation.ipynb))
- `mdata.mod['atac']` — cells × consensus peaks (raw counts), called by MACS3 per cell type; `tsse` and `n_fragment` QC metrics in `.obs`
- `mdata.uns['gene_tss']` — each gene's TSS (from the GTF). Defines the genomic window for peak-to-gene linking — no integration model needed.
- `mdata.uns['alphagenome_cache']` — pre-computed TF-binding predictions for the featured enhancer.

The same cell barcodes index both modalities: row *i* of RNA and row *i* of ATAC are the **same cell**.

In [ ]:
# What's actually inside the .h5mu we loaded?
print("=" * 60)
print("MODALITIES (per-cell, per-feature matrices)")
print("=" * 60)
for name in mdata.mod:
    m = mdata.mod[name]
    print(f"  mdata.mod['{name}']: {m.n_obs:,} cells x {m.n_vars:,} features")
    print(f"    obs columns: {list(m.obs.columns)[:8]}{'...' if len(m.obs.columns)>8 else ''}")
    print(f"    obsm keys:   {list(m.obsm.keys())}")

print()
print("=" * 60)
print("UNS (auxiliary data carried along)")
print("=" * 60)
for k, v in mdata.uns.items():
    if isinstance(v, dict):
        print(f"  mdata.uns['{k}']: dict with keys {sorted(v.keys())[:6]}{'...' if len(v)>6 else ''}")
    else:
        print(f"  mdata.uns['{k}']: {type(v).__name__}")

## Part 1: scATAC-seq + quality metrics (~5 min)

ATAC-seq uses the **Tn5 transposase** to cut DNA wherever it's accessible (not wrapped tightly into nucleosomes). The pile-ups of cuts that survive amplification become **peaks** — short genomic intervals (~500 bp) marking accessible chromatin. Each peak is a candidate *cis*-regulatory element: a promoter, an enhancer, an insulator.

**Two key per-cell quality metrics for ATAC:**

- **`n_fragment`** — how many sequencing fragments mapped to that cell's barcode. A cell with very few fragments (< ~1000) is likely barely sampled.
- **`tsse` (TSS enrichment)** — real chromatin signal piles up sharply at transcription start sites because TSSs are nearly always accessible in active cells. The ratio of fragment density at TSSs vs. flanking regions distinguishes real cells from ambient/dead barcodes. **TSSe ≥ 7 is the standard threshold.**

Both were applied as filters during preprocessing — the cells you see here are the ones that passed.

In [ ]:
# Quick ATAC modality summary + per-cell QC distributions
import scipy.sparse as sp
atac = mdata.mod['atac']
rna  = mdata.mod['rna']

print(f"ATAC matrix : {atac.n_obs:,} cells x {atac.n_vars:,} consensus peaks")
print(f"Sparsity    : {1 - atac.X.nnz / (atac.shape[0]*atac.shape[1]):.1%} of entries are zero")
print()
for col in ['n_fragment', 'tsse']:
    s = atac.obs[col]
    print(f"{col:>10}  median = {s.median():>10.2f}  min = {s.min():>10.2f}  max = {s.max():>10.2f}")


In [ ]:
# ATAC UMAP, three panels:
#   (1) cell type — labels transferred from the RNA annotation (same barcodes)
#   (2) TSS enrichment — sanity-check that signal isn't driving cluster structure
#   (3) n_fragment    — same check on library depth
# Cell types are clearly separated in ATAC alone — accessibility is informative
# about identity even without RNA.
import matplotlib.pyplot as plt
atac.obs['cell_type'] = atac.obs['cell_type'].astype('category')
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sc.pl.umap(atac, color="cell_type",   ax=axes[0], show=False,
           title="ATAC UMAP — cell type", legend_loc="right margin", legend_fontsize=8)
sc.pl.umap(atac, color="tsse",        ax=axes[1], show=False,
           title="TSS enrichment", cmap="viridis")
sc.pl.umap(atac, color="n_fragment",  ax=axes[2], show=False,
           title="Fragments per cell", cmap="viridis")
plt.tight_layout(); plt.show()

Two things to notice:

- **The ATAC UMAP recovers cell-type structure independently** of RNA — chromatin accessibility alone separates monocytes from T cells from B cells. Different cell types use different regulatory landscapes.
- **TSSe and n_fragment are diffuse across the UMAP**, not clumped in one cluster. That means cluster structure is being driven by biology (cell type), not by quality (signal depth). A clump of low-TSSe cells in one corner would have been a red flag.

## Part 2: RNA and ATAC are the same cells (~1 min)

Because this is *paired* multiome, "integrating" the two modalities is trivial: every ATAC barcode has a matching RNA barcode, so the cell-type labels we annotated on RNA transfer to ATAC by a simple lookup — which is exactly how the ATAC UMAP in Part 1 got its labels.

(Deep-learning methods like scGLUE exist for the *unpaired* case — different cells measured in each assay — but we don't need them here. The whole value of multiome is that the cells are matched, and that's precisely what powers the peak-to-gene analysis next.)

In [ ]:
# Same cells, two views: the RNA UMAP and the ATAC UMAP, each colored by the
# cell-type labels (annotated on RNA, carried to ATAC by shared barcode).
rna.obs['cell_type'] = rna.obs['cell_type'].astype('category')
atac.obs['cell_type'] = atac.obs['cell_type'].astype('category')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc.pl.umap(rna, color='cell_type', ax=axes[0], show=False,
           title='RNA UMAP (expression)', legend_loc='right margin', legend_fontsize=8)
sc.pl.umap(atac, color='cell_type', ax=axes[1], show=False,
           title='ATAC UMAP (accessibility, snapatac2)', legend_loc='right margin', legend_fontsize=8)
plt.tight_layout(); plt.show()
print('Two independent embeddings of the SAME cells — gene expression vs chromatin')
print('accessibility — carrying the same labels. Cell identity is written in both layers.')

## Part 3: Peak-to-gene at scale — DORCs (~15 min)

RNA and ATAC share the same cells, so we can ask, for any gene, **which nearby peaks have accessibility that tracks the gene's expression across cells** — candidate regulatory elements. This is the paired-multiome analysis at the heart of the workshop (the same statistic Signac `LinkPeaks` / ArchR `addPeak2GeneLinks` compute).

Run it for *every* gene, count each gene's significantly-correlated peaks, and a striking pattern appears: a small set of genes is linked to **many** enhancers. These are **DORCs — Domains of Regulatory Chromatin** (Kartha et al., 2022), and they are overwhelmingly **cell-identity / lineage-defining genes**. We precomputed the genome-wide DORC table; let's look at it.

In [ ]:
# DORC discovery was precomputed genome-wide (peak-gene correlation for every
# gene, peaks within +/-250kb of the TSS). We just load and visualize it.
dorc = pd.DataFrame({k: np.asarray(mdata.uns['dorc'][k])
                     for k in ['gene', 'n_cand', 'n_sig', 'max_r']})
cutoff = int(mdata.uns['dorc_params']['dorc_min_peaks'])
d = dorc.sort_values('n_sig', ascending=False).reset_index(drop=True)
n_dorc = int((d['n_sig'] >= cutoff).sum())
print(f"{len(d):,} genes scored; {n_dorc} are DORCs (>= {cutoff} correlated peaks)")
print('\nTop 15 DORCs (most enhancers):')
print(d.head(15)[['gene','n_cand','n_sig','max_r']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(len(d)), d['n_sig'], s=5, color='#bbbbbb')
isd = (d['n_sig'] >= cutoff).values
ax.scatter(np.where(isd)[0], d['n_sig'][isd], s=6, color='#d62728')
ax.axhline(cutoff, ls='--', color='k', lw=1)
for label in ['LEF1','PAX5','BCL11B','IRF8','CD8A','CD8B','MS4A1','SPI1','NKG7','TCF7','CD14']:
    hit = d.index[d['gene'] == label]
    if len(hit):
        i = hit[0]
        ax.annotate(label, (i, d['n_sig'][i]), fontsize=8, xytext=(3,2), textcoords='offset points')
ax.set_xlabel('genes ranked by # correlated enhancers')
ax.set_ylabel('# significantly correlated peaks')
ax.set_title(f'DORCs: {n_dorc} densely-regulated genes (red) vs the long tail')
plt.tight_layout(); plt.show()

### Zoom into one DORC: MS4A1

The DORC list is full of identity genes. Let's open up **MS4A1** (CD20), a textbook **B-cell** marker and a DORC, and look at its enhancers — every peak within ±250 kb of the TSS, ranked by how tightly its accessibility tracks MS4A1 expression across cells. We also tag each peak's B-vs-CD4 differential accessibility.

In [ ]:
# Normalize + log-transform the peak matrix, then differential accessibility
# between B cells and CD4 T cells (a contrasting type).
if "counts" not in atac.layers:
    atac.layers['counts'] = atac.X.copy()
sc.pp.normalize_total(atac, target_sum=1e4); sc.pp.log1p(atac)

sc.tl.rank_genes_groups(atac, groupby='cell_type', method='wilcoxon',
                        groups=['B cell'], reference='CD4 T cell')
da = sc.get.rank_genes_groups_df(atac, group='B cell')
da_score = dict(zip(da['names'], da['scores']))
print('Differential accessibility computed (B cell vs CD4 T).')

### The regulatory elements of MS4A1

Take every peak within **±250 kb of the MS4A1 TSS**, correlate each peak's accessibility with MS4A1 expression across all cells, and rank — annotated with **distance to the TSS** and B-vs-CD4 differential accessibility.

In [ ]:
# Peak-to-gene by paired correlation; candidate window from the TSS (gene_tss).
import re
import scipy.sparse as sp
from scipy import stats

_tss = mdata.uns['gene_tss']
TSS = {g: (c, int(t)) for g, c, t in zip(_tss['gene'], _tss['chrom'], _tss['tss'])}
_pc = atac.var_names.str.extract(r'(?P<chrom>chr\w+)[:\-](?P<start>\d+)[:\-](?P<end>\d+)')
PEAKS = pd.DataFrame({'name': atac.var_names, 'chrom': _pc['chrom'].values,
                      'mid': ((_pc['start'].astype(int)+_pc['end'].astype(int))//2).values})

def dense(ad, name):
    col = ad[:, name].X
    return np.asarray(col.todense()).ravel() if sp.issparse(col) else np.asarray(col).ravel()

def link_to_gene(gene, window=250_000):
    chrom, tss = TSS[gene]
    near = PEAKS[(PEAKS.chrom==chrom)&(PEAKS.mid>=tss-window)&(PEAKS.mid<=tss+window)].copy()
    g = dense(rna, gene); rhos=[]
    for nm in near['name']:
        a = dense(atac, nm)
        rhos.append(stats.spearmanr(a,g)[0] if a.std()>0 else np.nan)
    near['rho']=rhos
    near['dist_kb']=((near['mid']-tss)/1000).round(1)
    d=near['dist_kb'].abs()
    near['kind']=np.where(d<2,'promoter',np.where(d<10,'gene-proximal','distal'))
    near['B_DA']=near['name'].map(da_score).round(1)
    return near.sort_values('rho',ascending=False).reset_index(drop=True)

cd8a = None  # (previous example retired)
ms4a1 = link_to_gene('LEF1')
print(f"MS4A1: {len(ms4a1)} candidate peaks within \u00b1250 kb of the TSS\n")
print(ms4a1[['name','rho','dist_kb','kind','B_DA']].head(8).to_string(index=False))
FEATURED_GENE='MS4A1'
_distal = ms4a1[ms4a1['kind']=='distal'].reset_index(drop=True)
FEATURED_PEAK = (_distal.iloc[0]['name'] if len(_distal) else ms4a1.iloc[0]['name'])
_fr = ms4a1[ms4a1['name']==FEATURED_PEAK].iloc[0]
print(f"\nFeatured DISTAL enhancer: {FEATURED_PEAK} ({_fr['dist_kb']:+.1f} kb from TSS, rho={_fr['rho']:.2f})")

# cell-type groupings reused below
cts = rna.obs['cell_type'].astype(str).values
ct_order = list(rna.obs['cell_type'].cat.categories)
by_ct = {ct: np.where(cts==ct)[0] for ct in ct_order}

**Reading the table:** the **promoter** (top row, ρ≈0.62, ~0 kb) tracks MS4A1 most tightly — expected, since a promoter opens whenever its gene is on. The more interesting hit is the top **distal** peak: **~43 kb from the TSS** (ρ≈0.41), strongly B-vs-CD4 differential — a candidate B-cell **enhancer**. That's the one we feature. Next we look at the locus in a genome browser, then identify the TF acting there.

### Genome-browser view (IGV) at the MS4A1 locus

Per-cell-type pseudobulk ATAC tracks in an embedded IGV browser, plus the hg38 gene track — so you see **where MS4A1 is** and **the accessibility peaks**, including the B-specific distal enhancer ~43 kb away (gold region).

> **Loading the tracks:** igv.js reads bigWig via HTTP range requests, so the `tracks/` bigWigs must be served from a range-capable URL (e.g. a public GCS bucket, like the `.h5mu`). Set `TRACKS_URL` to that base URL. Local file paths don't work reliably in Jupyter/Colab.

In [ ]:
# Embedded IGV browser. Set TRACKS_URL to the hosted tracks/ base URL (GCS).
import igv_notebook, os
TRACKS = os.environ.get('TRACKS_URL', 'https://storage.googleapis.com/broad-p16-calico/festival-2026/tracks')
chrom, se = FEATURED_PEAK.split(':'); ps, pe = map(int, se.split('-'))
igv_notebook.init()
b = igv_notebook.Browser({
    'genome': 'hg38',
    'locus': 'chr11:60,430,000-60,520,000',
    'roi': [{'chr': chrom, 'start': ps-200, 'end': pe+200, 'name': 'MS4A1 enhancer'}],
})
for ct in ['B cell','CD4 T cell','CD8 T cell','NK cell','CD14 Monocyte','CD16 Monocyte','Dendritic cell']:
    safe = ct.replace(' ', '_')
    color = '#1f77b4' if ct == 'B cell' else '#999999'
    b.load_track({'name': ct, 'url': f'{TRACKS}/{safe}.bw', 'format': 'bigwig',
                  'type': 'wig', 'height': 32, 'color': color, 'autoscaleGroup': 'atac'})
print('B-cell track (blue) shows the enhancer peak; MS4A1 gene is in the refseq track.')

In [ ]:
# Pseudobulk: one point per cell type — featured-peak accessibility vs MS4A1 expression
gene_expr = dense(rna, FEATURED_GENE); peak_acc = dense(atac, FEATURED_PEAK)
x = np.array([peak_acc[by_ct[c]].mean() for c in ct_order])
y = np.array([gene_expr[by_ct[c]].mean() for c in ct_order])
rho_pb,_ = stats.spearmanr(x,y)
fig,ax=plt.subplots(figsize=(7,5)); colors=plt.get_cmap('tab10').colors
for i,c in enumerate(ct_order):
    ax.scatter(x[i],y[i],s=150,color=colors[i%10],label=c,edgecolor='black')
ax.set_xlabel(f'mean log-norm accessibility \u2014 {FEATURED_PEAK}')
ax.set_ylabel('mean log-norm MS4A1 expression')
ax.set_title(f'MS4A1 enhancer \u2014 pseudobulk Spearman \u03c1 = {rho_pb:.2f}')
ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=9)
plt.tight_layout(); plt.show()
print('Enhancer accessibility and MS4A1 expression both peak in B cells.')

### Where does the enhancer fire on the UMAP?

Color the ATAC UMAP by the **featured peak's accessibility** next to **MS4A1 expression** from RNA. Both should light up the B-cell population.

In [ ]:
atac.obs['featured_peak_acc'] = dense(atac, FEATURED_PEAK)
atac.obs['MS4A1_expr'] = dense(rna, 'MS4A1')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc.pl.umap(atac, color='featured_peak_acc', ax=axes[0], show=False,
           title=f'Accessibility \u2014 {FEATURED_PEAK}', cmap='magma')
sc.pl.umap(atac, color='MS4A1_expr', ax=axes[1], show=False,
           title='MS4A1 expression (RNA)', cmap='magma')
plt.tight_layout(); plt.show()

## Part 4: Which TF binds this enhancer? (~6 min)

We have a B-cell enhancer linked to MS4A1. To find its regulator **without assuming one**, we ask **AlphaGenome** — a deep-learning model that predicts transcription-factor binding directly from DNA sequence (no ATAC needed) — which TFs bind here. The predictions are pre-computed and baked into the `.h5mu`, so this runs without an API key.

### AlphaGenome: TF binding predicted from sequence

For a panel of immune TFs, AlphaGenome scores how strongly each is predicted to bind across the locus; we rank them by predicted binding at the enhancer.

> **Re-targeting:** the cell calls `show_tf_binding(FEATURED_PEAK)`. With an `ALPHA_GENOME_API_KEY` set, it queries **any** peak live — so if you change `FEATURED_GENE` earlier and re-run, this panel follows. Without a key it uses the baked cache (the default MS4A1 enhancer).

In [ ]:
# Which TFs bind the featured enhancer? AlphaGenome predicts it from sequence.
#   - with an API key (Colab secret or ALPHA_GENOME_API_KEY env): queries live for
#     ANY peak, so changing FEATURED_GENE above and re-running this cell just works.
#   - without a key: uses the baked cache (covers the default MS4A1 enhancer).
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get("ALPHA_GENOME_API_KEY")
except Exception:
    API_KEY = os.environ.get("ALPHA_GENOME_API_KEY")

# broad immune-TF panel + ENCODE biosamples with good CHIP-TF coverage
TF_PANEL = ["EBF1","PAX5","SPIB","TCF3","POU2F2","SPI1","CEBPA","CEBPB","IRF8",
            "RUNX3","ETS1","TBX21","EOMES","TCF7","LEF1","GATA3","BCL11A","CTCF"]
BIOSAMPLES = ["EFO:0002784","EFO:0002067","CL:0000236","CL:0000624","CL:0000625","CL:0001054"]
TOP_PLOT = 6   # show the top-N TFs as tracks (full ranking is printed)

def _tf_live(peak):
    from alphagenome.data import genome
    from alphagenome.models import dna_client
    chrom, se = peak.split(":"); s, e = map(int, se.split("-"))
    model = dna_client.create(API_KEY)
    ctx = genome.Interval(chrom, s, e).resize(dna_client.SEQUENCE_LENGTH_1MB)
    out = model.predict_interval(interval=ctx,
            requested_outputs=[dna_client.OutputType.CHIP_TF], ontology_terms=BIOSAMPLES)
    vals = out.chip_tf.values
    tf = out.chip_tf.metadata["transcription_factor"].astype(str).values
    pos = np.linspace(ctx.start, ctx.end, vals.shape[0])
    win = (pos >= s-2000) & (pos <= e+2000)
    best = {}                                   # one track per TF (strongest in peak)
    for i, t in enumerate(tf):
        if t not in TF_PANEL:
            continue
        sig = vals[win, i].max()
        if t not in best or sig > best[t][1]:
            best[t] = (i, sig)
    idx = [v[0] for v in best.values()]
    return pos, vals[:, idx], np.array(list(best)), s, e

def _tf_cache(peak):
    c = mdata.uns.get("alphagenome_cache", {})
    if c.get("featured_peak") != peak:
        return None
    vals = np.asarray(c["chip_tf_values"])
    md = pd.DataFrame({k: np.asarray(v) for k, v in c["chip_tf_metadata"].items()})
    pos = np.linspace(int(c["context_start"]), int(c["context_end"]), vals.shape[0])
    return pos, vals, md["transcription_factor"].astype(str).values, int(c["peak_start"]), int(c["peak_end"])

def show_tf_binding(peak):
    res, src = (_tf_live(peak), "live AlphaGenome") if API_KEY else (_tf_cache(peak), "baked cache")
    if res is None:
        print(f"No API key and no cache for {peak}.")
        print("Set ALPHA_GENOME_API_KEY to query any peak live, or keep the cached MS4A1 enhancer.")
        return
    pos, vals, names, ps, pe = res
    if vals.shape[1] == 0:
        print("No panel TFs returned for this peak."); return
    win = (pos >= ps-2000) & (pos <= pe+2000)
    rank = np.argsort(-vals[win].max(0))
    print(f"TF binding at {peak} ({src}), ranked by max signal in the peak:")
    for i in rank:
        print(f"  {names[i]:<8} {vals[win, i].max():.1f}")
    show = rank[:TOP_PLOT]
    fig, axes = plt.subplots(len(show), 1, figsize=(13, 1.6*len(show)), sharex=True, sharey=True)
    for ax, i in zip(np.atleast_1d(axes), show):
        ax.fill_between(pos, vals[:, i], alpha=0.7, color="#1f77b4")
        ax.axvspan(ps, pe, alpha=0.15, color="gray"); ax.set_ylabel(f"{names[i]}\nbinding")
        ax.set_xlim(ps-4000, pe+4000)
    axes[0].set_title(f"AlphaGenome predicted TF binding at {peak}  (top {len(show)})")
    plt.tight_layout(); plt.show()

show_tf_binding(FEATURED_PEAK)


**AlphaGenome predicts EBF1 and PAX5 — the master B-lineage transcription factors — bind this enhancer most strongly, from sequence alone.** We never told it which TF to look for.

The chain, end to end:

1. **DORC** — MS4A1 is a densely-regulated B-cell identity gene
2. **Peak-to-gene** — its top distal enhancer (~43 kb) is B-cell-specific (IGV view)
3. **TF binding** — AlphaGenome predicts the B-master TFs (EBF1/PAX5) bind it

A B-cell-accessible enhancer, linked to a B-cell gene, predicted to bind B-cell TFs — a coherent, data-driven regulatory picture, with no assumptions baked in.

*Caveat:* AlphaGenome's CHIP-TF tracks come from ENCODE cell lines (GM12878/K562); it predicts binding *compatibility* from sequence, not occupancy in your specific cells.

## Summary

| Step | Method | Output |
|---|---|---|
| ATAC QC | snapatac2 `tsse` / `n_fragment` | clean cell set |
| Same cells | RNA + ATAC UMAPs, shared labels | identity in both layers |
| **DORC discovery** | genome-wide paired peak-gene correlation | densely-regulated identity genes |
| Peak-to-gene zoom | per-gene correlation + TSS distance | MS4A1's distal B-cell enhancer |
| Browser view | per-cell-type pseudobulk bigWigs in IGV | enhancer peak + gene location, by cell type |
| TF binding | AlphaGenome CHIP-TF (sequence) | EBF1/PAX5 predicted to bind the enhancer |

### Core ideas

- **Paired multiome** lets you correlate peak accessibility with gene expression across the *same* cells — peak-to-gene linking and DORC discovery.
- **DORCs** surface lineage-defining genes automatically.
- A regulator can be **predicted from sequence** (AlphaGenome) without assuming which TF — and it lands on the right lineage TFs.

### Try it yourself
- Swap MS4A1 for another DORC (CD8A, LEF1, IRF8) in `link_to_gene(...)`, re-point the IGV locus, and (with an API key) re-run `show_tf_binding` — all three follow the new gene
- Ask AlphaGenome about a different locus's TFs

### Resources
- scanpy / snapatac2 · DORCs: Kartha et al. 2022 · igv-notebook: https://github.com/igvteam/igv-notebook
- AlphaGenome: https://www.alphagenomedocs.com · 10x Multiome: https://www.10xgenomics.com/datasets